# 06 — Explainability & explanation stability (Phase 5, RQ2/RQ3)

**DSP391m · Group 1 · FPT University** — the project's research contribution.

A recall of 0.93 is worthless to a tutor who cannot see *why* a student was flagged.
This notebook opens the model up and — the novel part — asks whether the explanation
is **trustworthy**:

- **RQ2 — seed stability:** if we retrain with a different random seed, does the
  explanation rank the same features? (It *should* — the data didn't change.)
- **RQ3 — checkpoint drift:** how does the explanation change as more of the course
  is observed? (Here drift is *expected and interesting*.)

All numbers are produced by `python -m tools.make_xai_analysis` (SHAP TreeExplainer +
LIME, both on the frozen split and the trained checkpoint bundles); this notebook
loads the committed tables/figures and interprets them.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
from IPython.display import Image, display

from src.config import FIGURES_DIR, TABLES_DIR


def show(name):
    display(Image(filename=str(FIGURES_DIR / f"{name}.png")))


print("tables:", TABLES_DIR)

## 1 · What drives an at-risk flag — global SHAP (RQ setup)

SHAP attributes each prediction back to the features, on the same transformed space
the model saw. The bar ranks mean |SHAP|; the beeswarm shows *direction* — e.g. many
**days since last activity** (red = high) pushes toward at-risk, while a high
**weighted score to date** (blue on the positive side) pulls away from it.

In [ ]:
shap_imp = pd.read_csv(TABLES_DIR / "xai_shap_importance.csv")
display(shap_imp.head(15).reset_index(drop=True))
show("shap_importance_xgb_t100")
show("shap_summary_xgb_t100")

## 2 · One student, one explanation — LIME + do SHAP & LIME agree?

LIME fits a local linear surrogate around a single student, giving a per-case "why".
Below is the most-confident at-risk student; red bars raise the risk, blue lower it.
We then check whether SHAP and LIME tell the *same* global story (top-10 Jaccard +
rank correlation) — agreement is a sanity check on either method.

In [ ]:
show("lime_local_example")

agree = pd.read_csv(TABLES_DIR / "xai_shap_vs_lime.csv")
display(agree)
print(
    "SHAP vs LIME top-10 Jaccard = {jaccard:.2f}, Spearman = {spearman:.2f}".format(
        **agree.iloc[0]
    )
)

**Reading it:** SHAP and LIME agree on the headline drivers but diverge on the mid-tier
ranking (moderate Jaccard/Spearman). That is a known effect — LIME's local linear fit
and SHAP's game-theoretic attribution optimise different things — and it is exactly
why we do not trust a single explainer blindly, motivating the stability analysis next.

## 3 · RQ2 — is the explanation stable across random seeds?

We retrain the model with several seeds and re-explain. A trustworthy explanation
should rank features almost identically each time (high mean Jaccard@10 and Spearman
across every seed pair).

In [ ]:
seeds = pd.read_csv(TABLES_DIR / "xai_stability_seeds.csv")
display(seeds)
print(
    "Across {n_pairs:.0f} seed pairs: mean Jaccard@10 = {mean_jaccard:.2f}, "
    "mean Spearman = {mean_spearman:.2f}".format(**seeds.iloc[0])
)

**Reading it:** the mean rank correlation is very high — the explanation is **stable to
the random seed**, so the feature story is a property of the data/model, not an artefact
of one training run. This is the trust guarantee an early-warning tool needs.

## 4 · RQ3 — how does the explanation drift across the course?

Now the change we *do* expect: as more of the course is observed (t = 10 → 100 %), the
drivers shift from mostly-demographic priors to concrete engagement/performance
signals. We measure the top-10 Jaccard and Spearman between checkpoints (consecutive
pairs, and each checkpoint vs the full-course reference).

In [ ]:
drift = pd.read_csv(TABLES_DIR / "xai_stability_checkpoints.csv")
display(drift)
show("xai_stability_drift")

**Reading it:** consecutive checkpoints agree strongly (high Spearman), so the
explanation evolves *smoothly* rather than jumping around — but the earliest views
(t = 10–20 %) differ markedly from the full-course view (low Jaccard vs t=100), which
is why an explanation shown to a tutor must be labelled with the checkpoint it came
from. Smooth, seed-stable, checkpoint-aware explanations are the project's RQ2/RQ3
contribution.

## Summary

- **SHAP** identifies days-since-last-activity and running score as the dominant
  at-risk drivers, with a directionally sensible beeswarm.
- **LIME** gives per-student explanations; SHAP↔LIME agree on the top drivers but not
  the full ranking — so we report both and test stability rather than trusting one.
- **RQ2:** explanations are **highly stable across seeds** (trustworthy).
- **RQ3:** explanations **drift smoothly** with course progress and diverge from the
  full-course view early on — actionable, and a caution for how they're presented.